# 多模态压缩与安全评估实践

[运行要求和入口](README.md)

先验证未量化缩放等价式，再运行量化/推理命令。模型、输入数据和 GPU 由本地提供。


In [ ]:
import torch
x, weight = torch.randn(4, 8), torch.randn(3, 8)
scale = torch.rand(8) + 0.5
torch.testing.assert_close(x @ weight.T, (x / scale) @ (weight * scale).T)
print('Equivalent scaling before quantization: passed')


## 准备校准文本

本地已保存原 val.jsonl.zst。下面通过 datasets 读取，固定种子抽样，导出短文本；应检查 tokenizer 长度，正式校准要求足够文本组成 512 token 块。这个单元格需要单独安装实践依赖，不会下载语料或启动模型。


In [ ]:
from pathlib import Path
from datasets import load_dataset
root = Path.cwd().resolve().parents[2]
source = root/'datasets/pile-val/val.jsonl.zst'
output = root/'outputs/mllm-compression-safety'
output.mkdir(parents=True, exist_ok=True)
if source.exists():
    dataset = load_dataset('json', data_files=str(source), split='train', cache_dir=str(output/'data-cache'))
    subset = dataset.shuffle(seed=42).select(range(min(2048, len(dataset))))
    subset.to_json(str(output/'calibration.jsonl'), force_ascii=False)
else:
    print('Prepare the local calibration corpus first; see README.')


### 实践：评估覆盖率与分母

继续 [多模态安全评估](README.md)。固定图像、问题 ID、生成设置，分别运行浮点和 AWQ 模型；再独立标注，不把缺失/失败的评审算作安全。

`metrics.summarize` 遍历问题和模型，分别统计 safe、unsafe、unjudged，报告 coverage = judged / total 和 unsafe_rate = unsafe / judged。覆盖率不同的两组不能直接比较为完整 benchmark 分数；没有标注时指标为 None。下面是合成标签练习，不是实际模型结果。


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
from metrics import summarize
toy = {
    '0': {'ans': {'demo': {'is_safe(gpt)': 'safe'}}},
    '1': {'ans': {'demo': {'is_safe(gpt)': 'unsafe'}}},
    '2': {'ans': {'demo': {}}},
}
result = summarize(toy)['demo']
assert result['unjudged'] == 1 and result['unsafe_rate'] == 0.5
print(result)
